# 02_Generative_AI_Working_Demo

## Workshop: From Predictive to Generative AI

### Learning Objectives:
- Experience how Generative AI approaches classification tasks
- Master the art of Prompt Engineering
- Compare cloud-based (Gemini) vs local (Ollama) AI solutions
- Understand robust AI system design with fallbacks

### The Challenge:
Can we solve the same Titanic prediction problem using natural language and AI conversation instead of traditional machine learning?

## Setup: Install Required Libraries

In [ ]:
# Install required libraries for Generative AI
# pip install google-generativeai requests pandas

# Import essential libraries
import google.generativeai as genai
import requests
import json
import pandas as pd
import re
from typing import Dict

print("✅ All libraries installed successfully!")
print("🚀 Ready to explore Generative AI!")

## The Art of Prompt Engineering

### 🎯 Key Insight: Simple, Clear Prompts Work Best!

We'll use a minimal prompt that's easy for AI models to follow.

In [ ]:
# Simple, effective prompt template
SIMPLE_PROMPT = """
Titanic passenger: {gender}, age {age}, class {pclass}, fare ${fare}

Predict survival. Reply with just:
SURVIVED or DIED
"""

print("📝 Simple Prompt Created!")
print("🎯 Key: Keep it simple and clear for AI models")

## Method 1: Google Gemini (Cloud-Based AI)

In [ ]:
def setup_gemini() -> bool:
    """
    Setup Google Gemini AI with API key.
    """
    try:
        print("🔑 Setting up Gemini API...")
        
        # ⚠️ DEMO ONLY: Replace with your actual API key
        api_key = "YOUR_API_KEY_HERE"
        
        if api_key == "YOUR_API_KEY_HERE":
            print("\n⚠️  Please add your Gemini API key!")
            print("   Get it from: https://makersuite.google.com/app/apikey")
            return False
        
        # Configure Gemini
        genai.configure(api_key=api_key)
        global gemini_model
        gemini_model = genai.GenerativeModel('gemini-2.5-flash')
        
        print("✅ Gemini setup successful!")
        return True
        
    except Exception as e:
        print(f"❌ Gemini setup failed: {str(e)}")
        return False

# Try to setup Gemini
gemini_ready = setup_gemini()

In [ ]:
def predict_with_gemini(passenger_data: Dict) -> Dict:
    """
    Use Gemini AI to predict Titanic survival with robust fallback.
    """
    try:
        # Format the simple prompt
        formatted_prompt = SIMPLE_PROMPT.format(
            gender=passenger_data['gender'],
            age=passenger_data['age'],
            pclass=passenger_data['pclass'],
            fare=passenger_data['fare']
        )
        
        print(f"🤖 Asking Gemini about: {passenger_data['gender']}, Age {passenger_data['age']}, Class {passenger_data['pclass']}")
        
        # Send to Gemini
        response = gemini_model.generate_content(formatted_prompt)
        response_text = response.text.strip().upper()
        
        print(f"   🔍 Gemini says: {response_text}")
        
        # Simple parsing - look for key words
        if "SURVIVED" in response_text:
            prediction = "SURVIVED"
            confidence = 0.8
        elif "DIED" in response_text:
            prediction = "DIED"
            confidence = 0.8
        else:
            # Fallback to simple rules
            prediction, confidence = simple_fallback_prediction(passenger_data)
            print(f"   ⚠️  Using fallback prediction")
        
        reasoning = f"Gemini AI analysis based on historical patterns"
        
        result = {
            "prediction": prediction,
            "confidence": confidence,
            "reasoning": reasoning
        }
        
        print(f"   → {result['prediction']} (Confidence: {result['confidence']:.1%})")
        return result
        
    except Exception as e:
        print(f"   ❌ Gemini failed: {str(e)}")
        # Use fallback prediction
        prediction, confidence = simple_fallback_prediction(passenger_data)
        return {
            "prediction": prediction,
            "confidence": confidence,
            "reasoning": "Fallback: Historical survival patterns"
        }

def simple_fallback_prediction(passenger_data: Dict) -> tuple:
    """
    Simple rule-based fallback when AI fails.
    """
    # Basic historical rules
    if passenger_data['gender'] == 'Female':
        if passenger_data['pclass'] <= 2:
            return "SURVIVED", 0.75  # Upper class women had high survival
        else:
            return "SURVIVED", 0.6   # Even 3rd class women had better odds
    else:  # Male
        if passenger_data['pclass'] == 1 and passenger_data['age'] < 16:
            return "SURVIVED", 0.7   # First class boys
        else:
            return "DIED", 0.7       # Most men didn't survive

# Test Gemini if ready
if gemini_ready:
    print("\n🧪 Testing Gemini...")
    test_passenger = {
        'pclass': 1, 'gender': 'Female', 'age': 25, 'fare': 100
    }
    gemini_result = predict_with_gemini(test_passenger)
    print("✅ Gemini test complete!")
else:
    print("⏭️  Skipping Gemini - API key needed")

## Method 2: Ollama (Local AI Server)

In [ ]:
def setup_ollama() -> bool:
    """
    Check if Ollama server is running locally.
    """
    try:
        print("🔍 Checking for local Ollama server...")
        response = requests.get("http://localhost:11434/api/tags", timeout=5)
        
        if response.status_code == 200:
            models = response.json().get('models', [])
            print(f"✅ Ollama found! Available models: {len(models)}")
            
            # Look for llama3 model specifically
            global ollama_model
            llama3_models = [m for m in models if 'llama3' in m['name'].lower()]
            if llama3_models:
                ollama_model = llama3_models[0]['name']
                print(f"✅ Using Llama3: {ollama_model}")
                return True
            elif models:
                ollama_model = models[0]['name']
                print(f"⚠️  Llama3 not found, using: {ollama_model}")
                return True
            else:
                print("⚠️  No models found")
                return False
        else:
            print(f"❌ Ollama server error: {response.status_code}")
            return False
            
    except requests.exceptions.ConnectionError:
        print("❌ Cannot connect to Ollama server")
        print("\n💡 This is expected in Google Colab!")
        print("   Ollama runs locally on your computer.")
        print("\n🏠 To use Ollama locally:")
        print("   1. Install: https://ollama.ai")
        print("   2. Run: ollama pull llama3")
        print("   3. Start: ollama serve")
        return False
    except Exception as e:
        print(f"❌ Ollama error: {str(e)}")
        return False

# Check Ollama availability

ollama_ready = setup_ollama()

In [ ]:
def predict_with_ollama(passenger_data: Dict) -> Dict:
    """
    Use Ollama (local AI) to predict Titanic survival.
    """
    try:
        # Format the same simple prompt
        formatted_prompt = SIMPLE_PROMPT.format(
            gender=passenger_data['gender'],
            age=passenger_data['age'],
            pclass=passenger_data['pclass'],
            fare=passenger_data['fare']
        )
        
        print(f"🦙 Asking Ollama about: {passenger_data['gender']}, Age {passenger_data['age']}, Class {passenger_data['pclass']}")
        
        # Send to Ollama
        ollama_request = {
            "model": ollama_model,
            "prompt": formatted_prompt,
            "stream": False,
            "options": {"temperature": 0.1}
        }
        
        response = requests.post(
            "http://localhost:11434/api/generate",
            json=ollama_request,
            timeout=30
        )
        
        if response.status_code == 200:
            ollama_response = response.json()
            response_text = ollama_response.get('response', '').strip().upper()
            
            print(f"   🔍 Ollama says: {response_text[:100]}...")
            
            # Simple parsing
            if "SURVIVED" in response_text:
                prediction = "SURVIVED"
                confidence = 0.8
            elif "DIED" in response_text:
                prediction = "DIED"
                confidence = 0.8
            else:
                # Fallback
                prediction, confidence = simple_fallback_prediction(passenger_data)
                print(f"   ⚠️  Using fallback prediction")
            
            result = {
                "prediction": prediction,
                "confidence": confidence,
                "reasoning": "Local Ollama AI analysis"
            }
            
            print(f"   → {result['prediction']} (Confidence: {result['confidence']:.1%})")
            return result
        else:
            print(f"   ❌ Ollama API error: {response.status_code}")
            raise Exception("API error")
            
    except Exception as e:
        print(f"   ❌ Ollama failed: {str(e)}")
        # Use fallback
        prediction, confidence = simple_fallback_prediction(passenger_data)
        return {
            "prediction": prediction,
            "confidence": confidence,
            "reasoning": "Fallback: Historical survival patterns"
        }

# Test Ollama if ready
if ollama_ready:
    print("\n🧪 Testing Ollama...")
    test_passenger = {
        'pclass': 3, 'gender': 'Male', 'age': 30, 'fare': 15
    }
    ollama_result = predict_with_ollama(test_passenger)
    print("✅ Ollama test complete!")
else:
    print("⏭️  Skipping Ollama - server not available")

## The Great AI Showdown: Gemini vs Ollama vs Fallback Rules

In [ ]:
# Test passengers for our showdown
test_passengers = [
    {
        'name': 'Lady Margaret (First Class)',
        'pclass': 1, 'gender': 'Female', 'age': 25, 'fare': 100
    },
    {
        'name': 'John Smith (Third Class)',
        'pclass': 3, 'gender': 'Male', 'age': 30, 'fare': 15
    },
    {
        'name': 'Mrs. Brown (Second Class)',
        'pclass': 2, 'gender': 'Female', 'age': 45, 'fare': 50
    },
    {
        'name': 'Young Tommy (Third Class)',
        'pclass': 3, 'gender': 'Male', 'age': 8, 'fare': 20
    }
]

print("🥊 AI SHOWDOWN: Gemini vs Ollama vs Rules")
print("=" * 60)

results = []

for passenger in test_passengers:
    print(f"\n👤 Testing: {passenger['name']}")
    print(f"   Details: {passenger['gender']}, Age {passenger['age']}, Class {passenger['pclass']}, Fare ${passenger['fare']}")
    
    # Test with Gemini
    if gemini_ready:
        gemini_result = predict_with_gemini(passenger)
        gemini_pred = f"{gemini_result['prediction']} ({gemini_result['confidence']:.1%})"
    else:
        gemini_pred = "Not Available"
    
    # Test with Ollama
    if ollama_ready:
        ollama_result = predict_with_ollama(passenger)
        ollama_pred = f"{ollama_result['prediction']} ({ollama_result['confidence']:.1%})"
    else:
        ollama_pred = "Not Available"
    
    # Test with simple rules
    rule_prediction, rule_confidence = simple_fallback_prediction(passenger)
    rules_pred = f"{rule_prediction} ({rule_confidence:.1%})"
    
    # Store results
    results.append({
        'Passenger': passenger['name'],
        'Gemini': gemini_pred,
        'Ollama': ollama_pred,
        'Rules': rules_pred
    })
    
    print(f"   🤖 Gemini: {gemini_pred}")
    print(f"   🦙 Ollama: {ollama_pred}")
    print(f"   📏 Rules: {rules_pred}")

# Final comparison table
print("\n📊 FINAL COMPARISON TABLE:")
comparison_df = pd.DataFrame(results)
print(comparison_df.to_string(index=False))

## Key Insights: Building Robust AI Systems

### 🎯 What We Learned:

#### **1. Prompt Engineering Best Practices:**
- **Keep it simple**: Complex prompts often fail
- **Be specific**: Ask for exactly what you want
- **Test iteratively**: Start simple, then refine

#### **2. Robust AI System Design:**
- **Always have fallbacks**: AI models can fail
- **Graceful degradation**: Simple rules as backup
- **Error handling**: Expect and handle failures

#### **3. Cloud vs Local AI Trade-offs:**

| Aspect | Gemini (Cloud) | Ollama (Local) | Rules (Fallback) |
|--------|----------------|----------------|------------------|
| **Reliability** | High (with internet) | Medium | Very High |
| **Speed** | Fast | Moderate | Instant |
| **Privacy** | Data sent to Google | Completely private | Completely private |
| **Cost** | Pay per request | Free after setup | Free |
| **Accuracy** | High (when working) | Good (when working) | Basic but consistent |

### 🚀 For Your Hackfest Projects:

**Choose Generative AI when you need:**
- Natural language understanding
- Creative problem solving
- Complex reasoning
- Rapid prototyping

**Always include fallbacks for:**
- Network failures
- API rate limits
- Model unavailability
- Unexpected responses

### 💡 Pro Tips:
1. **Start with simple prompts** - complexity can be added later
2. **Test with edge cases** - see how your system handles unusual inputs
3. **Monitor and log** - track what works and what doesn't
4. **Have multiple strategies** - combine AI with traditional approaches

**🌟 Remember**: The best AI systems are not just smart - they're reliable, robust, and handle failure gracefully!

## 🎯 Workshop Challenge: Your Turn!

### Experiment Ideas:

1. **Improve the Prompt**: Try different ways to ask for predictions
2. **Add More Context**: Include historical facts in your prompts
3. **Test Edge Cases**: What happens with unusual passenger profiles?
4. **Combine Approaches**: Use AI for reasoning, rules for reliability
5. **Build Your Own**: Apply this pattern to different problems

### 💡 Workshop Takeaways:
- **Prompt engineering is an art** - simple often beats complex
- **Robust systems have fallbacks** - always plan for failure
- **Different AI tools have different strengths** - choose wisely
- **The future is hybrid** - combine AI with traditional approaches

**Happy Hacking! 🚀**

## 💬 Bonus: Chat with AI About the Data

Ask the AI questions about the Titanic dataset and get insights!

In [ ]:
def chat_with_ai(question: str) -> str:
    """Ask AI questions about the Titanic data."""
    context = """You are analyzing the Titanic dataset (891 passengers).
    Key facts: 38% survival rate, Women: 74% survival, Men: 19% survival.
    Class survival: 1st: 63%, 2nd: 47%, 3rd: 24%."""
    
    prompt = f"{context}\n\nQuestion: {question}\n\nAnswer clearly:"
    
    try:
        if 'gemini_model' in globals() and gemini_ready:
            response = gemini_model.generate_content(prompt)
            return response.text.strip()
        elif 'ollama_model' in globals() and ollama_ready:
            ollama_request = {"model": ollama_model, "prompt": prompt, "stream": False}
            response = requests.post("http://localhost:11434/api/generate", json=ollama_request)
            if response.status_code == 200:
                return response.json().get('response', 'No response')
        
        # Fallback answers
        if 'women' in question.lower():
            return 'Women had 74% survival vs 19% for men due to "women and children first" policy.'
        elif 'class' in question.lower():
            return 'First class: 63% survival, Second: 47%, Third: 24%. Higher class = better lifeboat access.'
        else:
            return 'Survival depended on gender, class, and age. Women and children in higher classes had best chances.'
    except Exception as e:
        return f"Error: {str(e)}"

print("💬 Chat with AI about the Titanic data!")
print("Try: chat_with_ai('Why did more women survive?')")
print("Try: chat_with_ai('How did class affect survival?')")

In [ ]:
# Load Titanic data for analysis
try:
    url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    titanic_data = pd.read_csv(url)
    print(f"📊 Titanic data loaded: {titanic_data.shape[0]} passengers")
except:
    titanic_data = None
    print("⚠️  Could not load data. Using fallback responses.")

def get_real_stats():
    if titanic_data is None:
        return None
    
    df = titanic_data
    stats = {}
    stats['overall'] = df['Survived'].mean()
    stats['female'] = df[df['Sex'] == 'female']['Survived'].mean()
    stats['male'] = df[df['Sex'] == 'male']['Survived'].mean()
    stats['class_1'] = df[df['Pclass'] == 1]['Survived'].mean()
    stats['class_2'] = df[df['Pclass'] == 2]['Survived'].mean()
    stats['class_3'] = df[df['Pclass'] == 3]['Survived'].mean()
    
    df_age = df.dropna(subset=['Age'])
    stats['children'] = df_age[df_age['Age'] < 18]['Survived'].mean()
    stats['adults'] = df_age[df_age['Age'] >= 18]['Survived'].mean()
    return stats

def answer_with_data(question: str):
    stats = get_real_stats()
    if not stats:
        return None
    
    q = question.lower()
    if 'women' in q and ('men' in q or 'male' in q):
        return f"📊 REAL DATA: Women: {stats['female']:.1%} vs Men: {stats['male']:.1%} survival. 'Women and children first' policy worked!"
    elif 'class' in q:
        return f"📊 REAL DATA: 1st: {stats['class_1']:.1%}, 2nd: {stats['class_2']:.1%}, 3rd: {stats['class_3']:.1%}. Money bought survival."
    elif 'age' in q or 'children' in q:
        return f"📊 REAL DATA: Children: {stats['children']:.1%} vs Adults: {stats['adults']:.1%}. Kids had priority."
    elif 'survival' in q and 'rate' in q:
        return f"📊 REAL DATA: {stats['overall']:.1%} overall survival ({int(stats['overall']*891)}/891 passengers)."
    return None

print("📊 Data analysis ready! Choose your chat option:")


### Chat with Gemini

In [ ]:
def chat_with_gemini(question: str) -> str:
    """Chat using Gemini - tries real data first, then AI knowledge"""
    
    # First: Try to answer with actual data
    data_answer = answer_with_data(question)
    if data_answer:
        return data_answer
    
    # Second: Use Gemini's knowledge
    try:
        if 'gemini_model' in globals() and gemini_ready:
            prompt = f"""You are a Titanic expert. Answer this question clearly and concisely:
            
            Question: {question}
            
            Provide an informative answer based on historical knowledge:"""
            
            response = gemini_model.generate_content(prompt)
            return f"🤖 Gemini AI: {response.text.strip()}"
        else:
            return "⚠️  Gemini not available. Set up API key first."
    except Exception as e:
        return f"❌ Gemini error: {str(e)}"

print("💬 OPTION 1: Gemini Chat Ready!")
print("Usage: chat_with_gemini('your question here')")


### Chat with Ollama

In [ ]:
def chat_with_ollama(question: str) -> str:
    """Chat using Ollama - tries real data first, then AI knowledge"""
    
    # First: Try to answer with actual data
    data_answer = answer_with_data(question)
    if data_answer:
        return data_answer
    
    # Second: Use Ollama's knowledge
    try:
        if 'ollama_model' in globals() and ollama_ready:
            prompt = f"""You are a Titanic expert. Answer this question clearly and concisely:
            
            Question: {question}
            
            Provide an informative answer based on historical knowledge:"""
            
            ollama_request = {
                "model": ollama_model,
                "prompt": prompt,
                "stream": False,
                "options": {"temperature": 0.3}
            }
            
            response = requests.post("http://localhost:11434/api/generate", json=ollama_request, timeout=30)
            if response.status_code == 200:
                result = response.json().get('response', 'No response')
                return f"🦙 Ollama AI: {result.strip()}"
            else:
                return f"❌ Ollama API error: {response.status_code}"
        else:
            return "⚠️  Ollama not available. Start Ollama server first."
    except Exception as e:
        return f"❌ Ollama error: {str(e)}"

print("💬 OPTION 2: Ollama Chat Ready!")
print("Usage: chat_with_ollama('your question here')")


In [ ]:
# Example Usage
print("🔍 Testing both data and AI responses:\n")

# Data-driven question
print("1. Data question:")
print(chat_with_gemini('Why did more women survive than men?'))
print()


In [ ]:
# Knowledge-based question  
print("2. Knowledge question:")
print(chat_with_gemini('What was the Titanic disaster?'))
print()

print("💡 Try your own questions!")
print("Data: 'How did class affect survival?'")
print("Knowledge: 'What lessons can we learn?'")

In [ ]:
# Try it out!
response = chat_with_ai('Why did more women survive than men?')
print("🤖 AI Response:")
print(response)

In [ ]:
response = chat_with_ollama('What can you do?')
print("🤖 AI Response:")
print(response)

### Workshop Demo Questions

#### 1.Basic Data Insights

In [ ]:
# Basic Data Insights
response = chat_with_gemini("Why did more men survive than women?")
print("🤖 AI Response:")
print(response)



In [ ]:
response = chat_with_ai("How did passenger class affect survival rates?")
print("🤖 AI Response:")
print(response)

In [ ]:
response = chat_with_ai("What role did age play in survival?")
print("🤖 AI Response:")
print(response)

In [ ]:
response = chat_with_ai("Which was more important: gender or class?")
print("🤖 AI Response:")
print(response)

#### 2. Historical Context

In [ ]:
response = chat_with_ai("What was the 'women and children first' policy?")
print("🤖 AI Response:")
print(response)


In [ ]:
response = chat_with_ai("Why did first class passengers have better survival rates?")
print("🤖 AI Response:")
print(response)

In [ ]:
response = chat_with_ai("How did the ship's design affect evacuation?")
print("🤖 AI Response:")
print(response)

In [ ]:
response = chat_with_ai("What happened during the Titanic sinking?")
print("🤖 AI Response:")
print(response)

#### 3. Data Analysis Guidance

In [ ]:
response = chat_with_ai("What's the most surprising finding in this data?")
print("🤖 AI Response:")
print(response)


In [ ]:
response = chat_with_ai("How should I analyze family size and survival?")
print("🤖 AI Response:")
print(response)

In [ ]:
response = chat_with_ai("What visualizations would be most helpful?")
print("🤖 AI Response:")
print(response)

In [ ]:
response = chat_with_ai("What machine learning model should I use?")
print("🤖 AI Response:")
print(response)

#### 4. Comparative Analysis

In [ ]:
response = chat_with_ai("Compare survival rates between children and adults")
print("🤖 AI Response:")
print(response)


In [ ]:
response = chat_with_ai("Did wealthy passengers always survive more?")
print("🤖 AI Response:")
print(response)

In [ ]:
response = chat_with_ai("How did crew members' survival compare to passengers?")
print("🤖 AI Response:")
print(response)

In [ ]:
response = chat_with_ai("What patterns do you see in the data?")
print("🤖 AI Response:")
print(response)

#### 5. Practical Applications

In [ ]:
response = chat_with_ai("How can this analysis help modern safety planning?")
print("🤖 AI Response:")
print(response)


In [ ]:
response = chat_with_ai("What lessons can we learn for emergency protocols?")
print("🤖 AI Response:")
print(response)

In [ ]:
reponse = chat_with_ai("How would you improve ship safety based on this data?")
print("🤖 AI Response:")
print(response)

In [ ]:
response = chat_with_ai("What would you tell a modern cruise line?")
print("🤖 AI Response:")
print(response)

#### 6. Technical Deep Dive

In [ ]:
response = chat_with_ai("What features should I engineer from this data?")
print("🤖 AI Response:")
print(response)


In [ ]:
response = chat_with_ai("How should I handle missing values?")
print("🤖 AI Response:")
print(response)

In [ ]:
response = chat_with_ai("What's the best way to validate my model?")
print("🤖 AI Response:")
print(response)

In [ ]:
response = chat_with_ai("How can I explain my model's predictions?")
print("🤖 AI Response:")
print(response)

#### 7. Creative Questions

In [ ]:
response = chat_with_ai("If you were on the Titanic, what would increase your survival chances?")
print("🤖 AI Response:")
print(response)


In [ ]:
response = chat_with_ai("What would a survival guide for 1912 passengers look like?")
print("🤖 AI Response:")
print(response)

In [ ]:
response = chat_with_ai("How would social media have changed the disaster?")
print("🤖 AI Response:")
print(response)

In [ ]:
response = chat_with_ai("What questions would a journalist ask about this data?")
print("🤖 AI Response:")
print(response)